# In-Class Activity: Build a Basic Data Pipeline

**MATH 153 · Data Wrangling / Data Engineering**
Howard University · Department of Mathematics

---

### The situation

Founders Library records every book checkout and sends the circulation office **one file per day**.
Three days have arrived. The office wants a clean table they can query, plus a daily summary of
how many books went out by genre.

**Your job: build the ETL pipeline.** Same five components as the café example — only the data
is different.

| | |
|---|---|
| **Source** | `activity_data/checkouts_2027-03-0{2,3,4}.csv` and `activity_data/books.csv` |
| **Extract** | read the three daily files and stack them |
| **Transform** | fix `loan_days`, clean `patron_type`, join `books`, add a due date |
| **Load** | write to `library.db` and `checkout_summary.csv` |
| **Schedule** | (we'll just talk about this one) |

### How to work

Six tasks. Each has a cell with `# TODO` in it. Fill in the blanks and run the cell — each one
prints something you can check against the expected answer written underneath.

Work in pairs. Ask when you are stuck; everything you need is in the lecture notebook.

---
## Setup — run this first

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DATA = Path("activity_data")
print("files:", sorted(p.name for p in DATA.iterdir()))

## Look before you build

Always do this first. Two things in this file need fixing — find them before you read on.

In [ ]:
peek = pd.read_csv(DATA / "checkouts_2027-03-02.csv")
print(peek.shape)
peek.head()

Did you spot them?

1. **`loan_days` is text**, not a number — it says `"14 days"`. You cannot add `"14 days"` to a date.
2. **`patron_type` is spelled inconsistently** — `STAFF`, `Faculty`, `graduate`, `  Undergrad  `.

Those two are your Transform step, exactly like `unit_price` and `item` in the café example.

In [ ]:
books = pd.read_csv(DATA / "books.csv")
books

This is the **reference table** — it tells you each book's genre and replacement cost.
You will join it on in Task 4.

---
## Task 1 — Extract  *(5 min)*

Write a function that reads all three daily files and stacks them into one DataFrame.

**Hints:** `DATA.glob("checkouts_2027-*.csv")` finds the files. `sorted()` puts them in date order.
`pd.concat(frames, ignore_index=True)` stacks them.

In [ ]:
def extract(data_dir):
    """Read every daily checkout file and stack them into one DataFrame."""
    files = sorted(data_dir.glob("checkouts_2027-*.csv"))
    # TODO: read each file into a DataFrame and put them in a list called `frames`
    frames = ...
    # TODO: stack the list into one DataFrame and return it
    return ...


raw = extract(DATA)
print("rows:", len(raw))
raw.head(3)

**Expected: 82 rows.**

Remember the rule from the lecture — Extract changes nothing. No cleaning here.

---
## Task 2 — Transform, step 1: make `loan_days` a number  *(5 min)*

`"14 days"` needs to become `14`.

**Hint:** `.str.replace(" days", "", regex=False)` then `.astype(int)`.

In [ ]:
df = raw.copy()        # always work on a copy

# TODO: strip " days" off the end and convert the column to a whole number
df["loan_days"] = ...

print("before:", raw["loan_days"].iloc[0], "| after:", df["loan_days"].iloc[0])
print("dtype :", df["loan_days"].dtype)

**Expected:** the dtype should be `int64`, not `object`.

Also convert the date while you are here — a date stored as text is just a string.

In [ ]:
# TODO: convert checkout_date to a real datetime (the format is YYYY-MM-DD)
df["checkout_date"] = ...

print("dtype:", df["checkout_date"].dtype)

---
## Task 3 — Transform, step 2: clean `patron_type`  *(5 min)*

Four patron types are hiding behind many different spellings. Make them consistent.

**Hint:** the café example used `.str.strip()` followed by `.str.title()`.

In [ ]:
before = df["patron_type"].nunique()

# TODO: remove stray spaces and make the capitalisation consistent
df["patron_type"] = ...

print(f"spellings before: {before}")
print(f"types after     : {df['patron_type'].nunique()}")
print(sorted(df["patron_type"].unique()))

**Expected: 16 spellings become 4** — `Faculty`, `Graduate`, `Staff`, `Undergrad`.

**Why does this step have to come before the join?** Write your answer in one sentence:

> *your answer here*

---
## Task 4 — Transform, step 3: join the books, add a due date  *(7 min)*

Join `books` so every checkout knows its genre, then work out when each book is due back.

**Hints:** join on `book_id` with `how="left"`. For the due date:
`df["checkout_date"] + pd.to_timedelta(df["loan_days"], unit="D")`.

In [ ]:
# TODO: left-join the books reference table on book_id
df = ...

print("rows after the join:", len(df), " <- should still be 82")
print("rows with no matching book:", df["genre"].isna().sum(), " <- should be 0")

In [ ]:
# TODO: due_date = checkout_date + loan_days
df["due_date"] = ...

df[["checkout_id", "checkout_date", "title", "genre", "loan_days", "due_date"]].head()

**The two checks to make after every join** — and they are printed above, so look at them:

- the row count stayed at 82 (if it grew, `books.csv` has a duplicate `book_id`)
- nothing failed to match (if some did, your `book_id` values need cleaning too)

---
## Task 5 — Load  *(6 min)*

Build the daily summary, then write both outputs: a database table and a CSV.

In [ ]:
# TODO: one row per (date, genre), with a count of checkouts and the average loan length
summary = (df
           .assign(day=df["checkout_date"].dt.date)
           .groupby(["day", "genre"], as_index=False)
           .agg(checkouts=("checkout_id", "size"), avg_loan_days=("loan_days", "mean"))
           .rename(columns={"day": "checkout_date"}))
summary["avg_loan_days"] = summary["avg_loan_days"].round(1)
summary.head(8)

In [ ]:
def load(clean, summary, db_path="library.db", csv_path="checkout_summary.csv"):
    """Write the results where people will actually read them."""
    out = clean.copy()
    out["checkout_date"] = out["checkout_date"].dt.strftime("%Y-%m-%d")
    out["due_date"] = out["due_date"].dt.strftime("%Y-%m-%d")

    con = sqlite3.connect(db_path)
    # TODO: write `out` to a table called "checkouts".
    #       Use if_exists="replace" so running twice does not duplicate the data.
    ...
    con.close()

    summary.to_csv(csv_path, index=False)
    return len(out)


n = load(df, summary)
print(f"loaded {n} rows into library.db")

**Expected: 82 rows loaded.**

---
## Task 6 — Put it all together, and protect it  *(8 min)*

Wrap E, T and L in one function. Add a log line for each step and **two assertions** between
Transform and Load.

In [ ]:
def run_pipeline(data_dir=DATA):
    """Extract -> Transform -> Load for the library checkout feed."""
    books = pd.read_csv(data_dir / "books.csv")

    # --- E
    raw = extract(data_dir)
    print(f"  EXTRACT   {len(raw):>4} rows read")

    # --- T  (TODO: copy your four transform steps in here)
    clean = raw.copy()
    ...
    print(f"  TRANSFORM {len(clean):>4} rows cleaned")

    # --- CHECK  (TODO: write two assertions)
    # 1. every checkout matched a book in the reference table
    # 2. every loan is at least 1 day long
    ...

    # --- L
    summary = (clean
               .assign(day=clean["checkout_date"].dt.date)
               .groupby(["day", "genre"], as_index=False)
               .agg(checkouts=("checkout_id", "size"), avg_loan_days=("loan_days", "mean"))
               .rename(columns={"day": "checkout_date"}))
    load(clean, summary)
    print(f"  LOAD      {len(clean):>4} rows written")
    return clean, summary


clean, summary = run_pipeline()

### Now run it twice

A pipeline you cannot run twice is a pipeline nobody can trust.

In [ ]:
run_pipeline()
run_pipeline()

con = sqlite3.connect("library.db")
print("rows in the table:", pd.read_sql_query("SELECT COUNT(*) AS n FROM checkouts", con)["n"][0])
con.close()

**Expected: still 82, not 164 or 246.**

If you got 246, your `load` used `if_exists="append"`. Change it to `"replace"` and run again.

---
## Bonus — ask the database a question  *(if you have time)*

Your data is in SQL now. Use it.

In [ ]:
con = sqlite3.connect("library.db")

# TODO: which genre was checked out most? Show genre and the number of checkouts,
#       most popular first.
query = """
    SELECT ...
"""
pd.read_sql_query(query, con)

In [ ]:
# TODO: how many checkouts by each patron type?
pd.read_sql_query("""
    SELECT ...
""", con)

In [ ]:
con.close()
print("pipeline complete")

---
## Before you leave — three questions to discuss

1. **Scheduling.** Your pipeline reads every file in the folder. Tomorrow a fourth file arrives.
   What happens when you run `run_pipeline()` again? Is that what you want?

2. **Where do the checks belong?** You put two assertions between Transform and Load. What would
   go wrong if you put them at the very end, after `load()` had already run?

3. **One thing you would add.** You have a working pipeline. Name one thing it still needs before
   you would be comfortable letting it run unattended every morning.

### What you built today

```
extract()  ->  transform  ->  assert  ->  load()
   82 rows      82 rows       2 checks     library.db
                                           checkout_summary.csv
```

Four functions, two checks, and a result somebody else can use. That is a data pipeline.